# Dan 4 — Function Calling (Upotreba alata)

## Svrha ovog notebooka
LLM modeli su "odsječeni" od svijeta nakon što im se završi trening i ne znaju šta se dešava danas. Ovdje učimo kako modelu dati "oči i uši" spajajući ga na internet alate!

**Ovdje ćete naučiti:**
1. Kako opisati vaše funkcije (alate) modelu koristeći JSON.
2. Kako model umjesto običnog teksta može odlučiti da "pozove" vašu funkciju.
3. Kako izvršiti pravu HTTP skriptu za dohvat vremenske prognoze (Open-Meteo).
4. Kompletnu petlju: Vi pitate -> Model traži alat -> Vi pokrećete alat -> Vraćate podatke modelu -> Model vam daje smislen odgovor na osnovu podataka!

**Prerequisites:**
- Instalirane biblioteke iz prve ćelije.
- `.env` fajl sa `DEEPSEEK_API_KEY` (za ovo je potreban sposoban model).

**Kako pokrenuti:** Pokrećite ćelije jednu po jednu. U *KORAKU 5* pažljivo posmatrajte logove da vidite proces razmišljanja modela.

In [ ]:
# KORAK 1: Setup
import sys, os
sys.path.insert(0, '..')
import subprocess
subprocess.run(['pip', 'install', 'openai', 'httpx', 'python-dotenv', '--quiet'], check=True)
print('OK')

In [ ]:
# KORAK 2: Direktni Open-Meteo (bez AI)
from tools.weather import get_weather_forecast
import json

r = get_weather_forecast('Sarajevo', 3)
print(json.dumps(r, indent=2, ensure_ascii=False))

In [ ]:
# KORAK 3: Definicija alata (schema)
from main import TOOLS
import json
print(json.dumps(TOOLS, indent=2, ensure_ascii=False))

In [ ]:
# KORAK 4: Ručno izvršenje alata
from tools.weather import execute_tool
args = '{"city": "Mostar", "days": 2}'
print(execute_tool('get_weather_forecast', args))

In [ ]:
# KORAK 5: Puna function calling petlja
import os
from dotenv import load_dotenv
from openai import OpenAI
from main import DEFAULT_SYSTEM, TOOLS
from tools.weather import execute_tool

load_dotenv('../.env')
klijent = OpenAI(api_key=os.getenv('DEEPSEEK_API_KEY'), base_url='https://api.deepseek.com')
messages = [
    {'role': 'system', 'content': DEFAULT_SYSTEM},
    {'role': 'user', 'content': 'Uporedi prognozu za Sarajevo i Banja Luku sutra.'},
]

for i in range(3):
    r = klijent.chat.completions.create(model='deepseek-chat', messages=messages, tools=TOOLS, tool_choice='auto')
    msg = r.choices[0].message
    if not msg.tool_calls:
        print('Finalni odgovor:', msg.content)
        break
    messages.append(msg.model_dump(exclude_none=True))
    for tc in msg.tool_calls:
        print('Tool:', tc.function.name, tc.function.arguments)
        result = execute_tool(tc.function.name, tc.function.arguments)
        messages.append({'role': 'tool', 'tool_call_id': tc.id, 'content': result})

In [ ]:
# KORAK 6: Cache demonstracija
import time
t0 = time.time()
get_weather_forecast('Tuzla', 3)
print(f'Prvi poziv: {time.time()-t0:.2f}s, iz_cachea={get_weather_forecast("Tuzla", 3).get("iz_cachea")}')